# Tome 0, Chapter 4: Inequalities (Python companion)

Exercises first, solutions at the end. Computer-science threads: **random testing** (hunting counterexamples
before proving), **cosine similarity** (Cauchy–Schwarz in search engines), and **entropy and compression**
(Jensen behind Huffman codes).

In [ ]:
import math
import heapq
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Exercises

**P4.1 (★) Hunt before you prove.** Three of these four inequalities hold for all $a, b, c > 0$; one is false. Find it by random testing, and print a counterexample.
1. $a^2+b^2+c^2 \ge ab+bc+ca$
2. $a^3+b^3 \ge a^2b+ab^2$
3. $(a+b+c)^2 \ge 4(ab+bc+ca)$
4. $\frac ab+\frac bc+\frac ca \ge 3$

In [ ]:
# Your code here


**P4.2 (★) Jensen, seen.** Plot $\exp$ on $[-1, 2]$ with the chord between two points, and mark $f(\lambda x+(1-\lambda)y)$ and $\lambda f(x)+(1-\lambda)f(y)$ for $\lambda = 0.3$.

In [ ]:
# Your code here


**P4.3 (★★) CS application: cosine similarity.** Search engines compare documents by the angle between their word-count vectors, $\cos\theta = \frac{\langle u, v\rangle}{\|u\|\,\|v\|}$, which lies in $[-1, 1]$ precisely by Cauchy–Schwarz. Compute the similarity matrix of the four short texts given in the solution, and say which pair is closest.

In [ ]:
# Your code here


**P4.4 (★★) CS application: entropy and Huffman codes.** For a probability distribution $p$, the entropy is $H(p) = -\sum p_i \log_2 p_i$.
1. Use Jensen to show $H(p) \le \log_2 n$, and check it numerically.
2. Build a Huffman code for the letter frequencies of a text, and compare its average code length $L$ with $H$. Shannon's theorem says $H \le L < H+1$.

In [ ]:
# Your code here


**P4.5 (★★★) Carleman's constant is sharp.** For $a_k = 1/k$ when $k \le N$ and $0$ after, compute the ratio $\sum_n (a_1\cdots a_n)^{1/n} / \sum_n a_n$ for $N$ up to $10^7$ (use `math.lgamma` to avoid overflow). Watch it creep up to $e$.

In [ ]:
# Your code here


---
## Solutions

### P4.1

In [ ]:
tests = {
    "1": lambda a, b, c: a*a + b*b + c*c >= a*b + b*c + c*a - 1e-12,
    "2": lambda a, b, c: a**3 + b**3 >= a*a*b + a*b*b - 1e-12,
    "3": lambda a, b, c: (a + b + c)**2 >= 4 * (a*b + b*c + c*a) - 1e-12,
    "4": lambda a, b, c: a/b + b/c + c/a >= 3 - 1e-12,
}
samples = rng.uniform(0.01, 10, size=(10_000, 3))
for name, ok in tests.items():
    bad = next((s for s in samples if not ok(*s)), None)
    print(name, "holds on all samples" if bad is None else f"FALSE, e.g. a, b, c = {np.round(bad, 3)}")

Inequality 3 fails at $a=b=c$ ($9 \ge 12$ is false). The true statement is $(a+b+c)^2 \ge 3(ab+bc+ca)$, equivalent to inequality 1. Random testing proves nothing, but it saves hours spent proving false statements.

### P4.2

In [ ]:
f = np.exp
x, y, lam = -0.5, 1.7, 0.3
t = np.linspace(-1, 2, 300)
m = lam * x + (1 - lam) * y
plt.plot(t, f(t), label="exp")
plt.plot([x, y], [f(x), f(y)], "o-", label="chord")
plt.plot([m], [f(m)], "s", label=r"$f(\lambda x+(1-\lambda)y)$")
plt.plot([m], [lam * f(x) + (1 - lam) * f(y)], "^", label=r"$\lambda f(x)+(1-\lambda)f(y)$")
plt.legend(); plt.title("Jensen: the chord lies above the graph"); plt.show()

### P4.3

In [ ]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "prime numbers are the atoms of arithmetic",
    "every integer is a product of prime numbers",
]
vocab = sorted({w for d in docs for w in d.split()})
V = np.array([[Counter(d.split())[w] for w in vocab] for d in docs], dtype=float)
norms = np.linalg.norm(V, axis=1)
S = V @ V.T / np.outer(norms, norms)
print(np.round(S, 2))
i, j = max(((i, j) for i in range(4) for j in range(i + 1, 4)), key=lambda ij: S[ij])
print("closest pair:", docs[i], "|", docs[j])

### P4.4

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

for n in [2, 5, 26]:
    for _ in range(1000):
        p = rng.dirichlet(np.ones(n))
        assert entropy(p) <= math.log2(n) + 1e-12
print("H(p) <= log2(n) on 3000 random distributions")

def huffman(freqs):
    heap = [(f, i, {s: ""}) for i, (s, f) in enumerate(freqs.items())]
    heapq.heapify(heap)
    k = len(heap)
    while len(heap) > 1:
        f1, _, c1 = heapq.heappop(heap)
        f2, _, c2 = heapq.heappop(heap)
        merged = {s: "0" + c for s, c in c1.items()} | {s: "1" + c for s, c in c2.items()}
        heapq.heappush(heap, (f1 + f2, k, merged)); k += 1
    return heap[0][2]

text = ("mathematics is the queen of the sciences and number theory is the queen of mathematics "
        "said gauss and he was right about many things")
freqs = Counter(text)
code_ = huffman(freqs)
total = sum(freqs.values())
p = np.array([freqs[s] / total for s in freqs])
L = sum(freqs[s] / total * len(code_[s]) for s in freqs)
print(f"entropy H = {entropy(p):.3f} bits/char, Huffman L = {L:.3f}, fixed-length = {math.ceil(math.log2(len(freqs)))}")

Why $H \le \log_2 n$: by Jensen for the concave $\log_2$, $H = \sum p_i \log_2 \frac{1}{p_i} \le \log_2 \sum p_i \cdot \frac1{p_i} = \log_2 n$, with equality for the uniform distribution. The lower bound $L \ge H$ comes from the same Jensen argument combined with Kraft's inequality $\sum 2^{-\ell_i} \le 1$ for prefix codes.

### P4.5

In [ ]:
def carleman_ratio(N):
    n = np.arange(1, N + 1)
    lhs = np.exp(-np.array([math.lgamma(k + 1) for k in n]) / n).sum()   # (1/n!)^(1/n)
    rhs = (1.0 / n).sum()
    return lhs / rhs

for N in [10, 10**3, 10**5, 10**7]:
    print(f"N = {N:>8}: ratio = {carleman_ratio(N):.4f}   (e = {math.e:.4f})")

The ratio increases towards $e$ (very slowly: the gap is of order $\frac{1}{\ln N}$, because both sums grow like $\ln N$), so no constant smaller than $e$ works in Carleman's inequality.